In [17]:
import pandas as pd

In [18]:
import numpy as np

In [19]:
import matplotlib.pyplot as plt

In [20]:
from sklearn.decomposition import TruncatedSVD

In [21]:
df = pd.read_csv("/Users/senuja/Jupyter Notebook/Video_Game_Sales_Predictor/data/external/5. Feature Selection/Feature Selection(1-hot).csv")

In [22]:
df.head()

,Game Title,Critic Score,Sales,North American Sales,Japanese Sales,EU Sales,Other Sales,Release Year,Genre_Action-Adventure,Genre_Adventure,...,Developer_Vicarious Visions,Developer_Visual Concepts,Decade,NA Share,EU Share,JP Share,Other Share,Is Recent,Game Quality,Game Quality Encoded
0,Grand Theft Auto V,0.933333,1.000000,0.652664,0.464789,1.000000,1.000000,2013,0,0,...,0,0,2010,0.313330,0.484506,0.048697,0.153468,0,High,2
1,Grand Theft Auto V,0.966667,0.953740,0.620902,0.281690,0.985787,0.967949,2014,0,0,...,0,0,2010,0.312532,0.500774,0.030944,0.155750,0,High,2
2,Grand Theft Auto: Vice City,0.955556,0.794291,0.861680,0.220657,0.557360,0.570513,2002,0,0,...,0,0,2000,0.520743,0.339938,0.029102,0.110217,0,High,2
3,Grand Theft Auto V,0.677988,0.780512,0.928279,0.028169,0.541117,0.455128,2013,0,0,...,0,0,2010,0.570888,0.335854,0.003781,0.089477,0,High,2
4,Call of Duty: Black Ops 3,0.788889,0.742126,0.633197,0.192488,0.614213,0.782051,2015,0,0,...,0,0,2010,0.409543,0.400928,0.027170,0.161696,1,High,2


In [23]:
df.dtypes # check data types

Game Title               object
Critic Score            float64
Sales                   float64
North American Sales    float64
Japanese Sales          float64
                         ...   
JP Share                float64
Other Share             float64
Is Recent                 int64
Game Quality             object
Game Quality Encoded      int64
Length: 133, dtype: object

In [24]:
# 1) Build numeric matrix (drop target), clean, and make floats
X = df.drop(columns=["Sales"]).select_dtypes(include=["number", "bool"]).copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X = X.astype(float)

In [25]:
# 2) Find K to hit ~90% variance
max_k = min(50, X.shape[1] - 1) if X.shape[1] > 1 else 1
svd_probe = TruncatedSVD(n_components=max_k, random_state=42)
svd_probe.fit(X)
cumvar = svd_probe.explained_variance_ratio_.cumsum()

# Auto-pick K
K = int(np.searchsorted(cumvar, 0.90) + 1) if X.shape[1] > 1 else 1
K = max(2, min(K, max_k))
print("Chosen components K =", K, "| Cumulative variance ≈", round(cumvar[K-1], 4))

Chosen components K = 2 | Cumulative variance ≈ 0.9581


In [27]:
# 3) Final SVD transform
svd = TruncatedSVD(n_components=K, random_state=42)
Z = svd.fit_transform(X)
comp_cols = [f"SVD_{i+1}" for i in range(K)]

In [28]:
# 4) OVERWRITE df with reduced components + Sales
df = pd.concat(
    [df[["Sales"]].reset_index(drop=True),
     pd.DataFrame(Z, columns=comp_cols)],
    axis=1
)

print("AFTER  (rows, cols):", df.shape)
display(df.head())

AFTER  (rows, cols): (17567, 3)


,Sales,SVD_1,SVD_2
0,1.000000,2844.692241,1.314958
1,0.953740,2845.400196,0.585634
2,0.794291,2829.842199,1.995664
3,0.780512,2844.692198,1.308442
4,0.742126,2846.108077,-0.145832


In [29]:
df.head()

# SVD_1 - a weighted mix of many original features (genre, console, publisher, critic score, etc.) 
        # chosen to capture the largest possible variance

#SVD_2 - another mix of features, capturing the second-most variance

,Sales,SVD_1,SVD_2
0,1.000000,2844.692241,1.314958
1,0.953740,2845.400196,0.585634
2,0.794291,2829.842199,1.995664
3,0.780512,2844.692198,1.308442
4,0.742126,2846.108077,-0.145832


In [30]:
df.to_csv("/Users/senuja/Jupyter Notebook/Video_Game_Sales_Predictor/data/external/6. Dimensionality Reduction/Dimensionality Reduction.csv", index=False)